In [1]:
# =====================================================================
# 0. 셋업
# - Qdrant에서 청크 가져오기 -> 직접 LLM으로 테스트셋 생성(ragas.testset 미사용)
# -> LangGraph 챗봇 실행 -> ragas.evaluate로 평가
#
# 필요 패키지: pip install ragas langchain-openai langchain-qdrant qdrant-client pandas
# =====================================================================

import os
import sys
import types
import random
import json

import pandas as pd
from qdrant_client import QdrantClient
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI


c:\3번쨰 플젝\SKN31-3rd-2Team\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------------------------------------------------------
# ragas 임포트 버그 우회
# ragas가 import될 때 langchain_community.chat_models.vertexai 를
# 무조건 불러오려고 하는데, 최신 langchain-community에는 이 경로가 삭제되어
# ModuleNotFoundError가 날 수 있음. 아래 블록이 이를 자동으로 우회한다.
# (langchain-community<0.4로 맞춰뒀다면 이 블록은 그냥 아무 일도 안 함)
# ---------------------------------------------------------------
try:
    import langchain_community.chat_models.vertexai  # noqa: F401
except ModuleNotFoundError:
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")

    class _ChatVertexAIStub:
        def __init__(self, *args, **kwargs):
            raise RuntimeError(
                "ChatVertexAI는 사용하지 않는 스텁(stub)입니다. "
                "Vertex AI를 실제로 쓰려면 langchain-google-vertexai를 설치하세요."
            )

    _stub.ChatVertexAI = _ChatVertexAIStub
    sys.modules["langchain_community.chat_models.vertexai"] = _stub
except ImportError:
    pass

from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)
from langchain_openai import OpenAIEmbeddings


C:\Users\박동관\AppData\Local\Temp\ipykernel_54448\2075231208.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community.chat_models.vertexai  # noqa: F401
C:\Users\박동관\AppData\Local\Temp\ipykernel_54448\2075231208.py:30: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\박동관\AppData\Local\Temp\ipykernel_54448\2075231208.py:30: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Us

In [3]:
# ---------------------------------------------------------------
# 1. Qdrant에서 청크 가져오기
# ---------------------------------------------------------------
def fetch_chunks_from_qdrant(
    collection_name: str,
    limit: int = 100,
    url: str = "http://localhost:6333",
) -> list[Document]:
    """Qdrant 컬렉션에서 저장된 텍스트 청크를 LangChain Document로 변환."""
    print(f"🔄 Qdrant '{collection_name}' 컬렉션에서 데이터를 가져오는 중...")
    client = QdrantClient(url=url)

    records, _ = client.scroll(
        collection_name=collection_name,
        limit=limit,
        with_payload=True,
        with_vectors=False,
    )

    docs = []
    for record in records:
        payload = record.payload or {}
        # langchain-qdrant 저장 방식에 따라 키가 달라질 수 있어 순서대로 fallback
        page_content = (
            payload.get("page_content")
            or payload.get("content")
            or payload.get("text")
        )
        metadata = payload.get("metadata") or {}

        if page_content:
            docs.append(Document(page_content=page_content, metadata=metadata))

    print(f"✅ 성공적으로 {len(docs)}개의 텍스트 청크를 로드했습니다.")
    return docs


In [ ]:
# ---------------------------------------------------------------
# 2. ragas TestsetGenerator로 테스트셋 생성
# query_distribution을 지정하지 않으면 ragas가 내부 기본 분포로
# simple / reasoning / multi_context 등 질문 유형을 알아서 섞어 생성한다
# ---------------------------------------------------------------
def generate_testset(
    docs: list[Document],
    testset_size: int = 5,
) -> pd.DataFrame:
    generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
    generator_embeddings = LangchainEmbeddingsWrapper(
        OpenAIEmbeddings(model="text-embedding-3-large")
    )

    generator = TestsetGenerator(
        llm=generator_llm,
        embedding_model=generator_embeddings,
        llm_context="""
너는 대한민국 군인 복지 및 혜택에 대한 질문 데이터셋을 생성하는 AI 전문가이다.
제공된 문서(Docs)의 내용을 기반으로, 현역 장병이나 군 입대 예정자들이 실제로 궁금해할 법한
현실적이고 구체적인 질문과 정확한 답변 쌍을 생성해라.
모든 질문과 답변은 반드시 자연스러운 한국어로 작성해야 한다.
""",
    )

    testset = generator.generate_with_chunks(
        chunks=docs,
        testset_size=testset_size,
    )

    testset_df = testset.to_pandas()
    testset_df.to_csv("qdrant_testset.csv", index=False, encoding="utf-8-sig")
    print(f"✅ 테스트셋 생성 완료 ({len(testset_df)}개) -> qdrant_testset.csv 저장")
    return testset_df


In [32]:
# ---------------------------------------------------------------
# 3. LangGraph 챗봇 실행해서 답변/컨텍스트 수집
# bot.ask(question) -> (references, answer) 형태를 가정
# ⚠️ 아래 import 경로는 실제 프로젝트 구조에 맞게 수정하세요.
# ---------------------------------------------------------------
from run_chatbot import LangGraphChatbot


def _extract_context_text(ref) -> str:
    """bot.ask()가 반환하는 references 원소를 문자열로 변환.
    Document / dict / str / 커스텀 객체 모두 대응."""
    if isinstance(ref, str):
        return ref
    if isinstance(ref, Document):
        return ref.page_content
    if isinstance(ref, dict):
        return ref.get("page_content") or ref.get("content") or ref.get("text") or str(ref)
    if hasattr(ref, "page_content"):
        return ref.page_content
    return str(ref)


def run_chatbot_on_testset(testset_df: pd.DataFrame) -> list[SingleTurnSample]:
    bot = LangGraphChatbot(verbose=False)
    samples = []

    for i, row in testset_df.iterrows():
        question = row["user_input"]
        reference = row.get("reference")

        references, answer = bot.ask(question)
        retrieved_contexts = [_extract_context_text(r) for r in references]

        samples.append(
            SingleTurnSample(
                user_input=question,
                response=answer,
                retrieved_contexts=retrieved_contexts,
                reference=reference,
            )
        )
        print(f"✅ [{i+1}/{len(testset_df)}] {question[:40]}...")

    return samples


In [33]:
# ---------------------------------------------------------------
# 4. ragas 평가
# faithfulness / answer_relevancy / context_precision / context_recall
# 뒤 두 개는 reference(정답)가 있어야 계산 가능
# ---------------------------------------------------------------
def run_evaluation(samples: list[SingleTurnSample]):
    eval_dataset = EvaluationDataset(samples=samples)

    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
    evaluator_embeddings = LangchainEmbeddingsWrapper(
        OpenAIEmbeddings(model="text-embedding-3-large")
    )

    metrics = [
        Faithfulness(),
        AnswerRelevancy(),
        LLMContextPrecisionWithReference(),  # reference 필요
        LLMContextRecall(),                  # reference 필요
    ]

    result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    return result


In [34]:
# ---------------------------------------------------------------
# 5. 설정값 (필요하면 여기만 수정)
# ---------------------------------------------------------------
COLLECTION_NAME = "guidance_vectordb"
QDRANT_URL = "http://localhost:6333"
CHUNK_LIMIT = 100
TESTSET_SIZE = 10


In [35]:
# 1) 청크 로드
docs = fetch_chunks_from_qdrant(
    collection_name=COLLECTION_NAME, limit=CHUNK_LIMIT, url=QDRANT_URL
)
len(docs)

🔄 Qdrant 'guidance_vectordb' 컬렉션에서 데이터를 가져오는 중...
✅ 성공적으로 100개의 텍스트 청크를 로드했습니다.


100

In [36]:
# 2) 테스트셋 생성 (ragas.testset 미사용)
testset_df = generate_testset(docs, testset_size=TESTSET_SIZE)
testset_df.head()


C:\Users\박동관\AppData\Local\Temp\ipykernel_54448\3229229012.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
C:\Users\박동관\AppData\Local\Temp\ipykernel_54448\3229229012.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(
Applying SummaryExtractor:   0%|          | 0/100 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/100 [00:00<?, ?it/s]Node 0e5a83b9-dd7b-4df3-8038-6497e291aa6b does not have a summary. Skipping filtering.
Node b975c25d-f2f5-4f40-ad83-df6c74fb6236 does not have a summary. Skipping filtering.
Node 6b0d0b0c-54b1-42b6-9cb5-1d2a60634923 does not have a summary. Skipping filtering.
Node 455c092f-e2a0-4025-999d-0c6787544a03 does not have a summary. Skipping filtering.
Node d122c4e4-8c7e-4ce2-b6c0-152c957ee60e does not have a summary. Skipping filtering.
Node 33f58e3e-cb57-413a-9751-a5cb53ebe52c does not have a summary. Skipping filtering.
Node 19960813-4d70-4c68-9373-b2a9a567149a does not have a summary. Skipping filtering.
Node 73b0f021-812f-4895-9b67-322fb64ace26 does not have a summary. Skipping filtering.
Node 28003f1c-da5c-4aa9-b9b1-73bfc5808bc1 does not have a summary. Skipping filtering.
Node 95bce66a-e549-455e-8aed-0624b61c9c6a does not have a summary. Skipping filtering.
Node bc225b5f-fe28-4017-a3a3-81109548789f does not have a summar

✅ 테스트셋 생성 완료 (11개) -> qdrant_testset.csv 저장


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,병 복지의 혜택은 무엇인가요?,[길라잡이\n길라잡이\n병 복지\n병 복지혜택 이렇게 다양합니다\n2026],병 복지혜택 이렇게 다양합니다.,Military Support Coordinator,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
1,경상국립대에서 제공하는 건강검진의 이용대상은 누구인가요?,[02\nPart\n생\n활\n편\n의\n \n분\n야\n51 \n2026 초급간부...,경상국립대에서 제공하는 건강검진의 이용대상은 현역/예비역/兵 회원 및 회원 가족(배...,Military Support Coordinator,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,차량야전정비병의 역할은 무엇인가요?,"[장갑차야전정비병, 전차야전정비병, 조리병, 중형차량운전병, 지게차운전병, \n차량...","차량야전정비병은 군에서 차량의 유지보수 및 정비를 담당하는 병과로, 다양한 군용 차...",Military Support Coordinator,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
3,해군의 장기복무 선발 혜택은 무엇인가요?,"[이 표는 **장기복무 선발 길라잡이**로, 장기복무 선발의 **혜택**과 **군별...","해군의 장기복무 선발 혜택으로는 소령(상사) 진출 기회, 계급별 정년(소령 50세,...",Military Health Coordinator,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
4,국외군사교육과 국내·외 학위과정 지원은 어떻게 전문인력 양성을 위한 기회를 제공하나요?,[<1-hop>\n\n01\nPart\n자\n기\n개\n발\n \n및\n \n전\n...,국외군사교육은 최신 군사지식을 습득하고 무관요원 및 권역별 지역전문가 양성을 위해 ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer


In [37]:
# 3) LangGraph 챗봇으로 답변/컨텍스트 수집
samples = run_chatbot_on_testset(testset_df)


✅ [1/11] 병 복지의 혜택은 무엇인가요?...
✅ [2/11] 경상국립대에서 제공하는 건강검진의 이용대상은 누구인가요?...
✅ [3/11] 차량야전정비병의 역할은 무엇인가요?...
✅ [4/11] 해군의 장기복무 선발 혜택은 무엇인가요?...
✅ [5/11] 국외군사교육과 국내·외 학위과정 지원은 어떻게 전문인력 양성을 위한 기회...
✅ [6/11] 취업컨설팅이 자기개발에 어떻게 도움을 줄 수 있나요?...
✅ [7/11] 2026 병 복지 길라잡이에서 제공하는 군 e-러닝 학습 콘텐츠와 특기병...
✅ [8/11] 병역판정검사와 관련하여 나라사랑카드의 주요 기능은 무엇이며, 이 카드가 ...
✅ [9/11] 인천에서 군인 및 군무원 가족을 위한 항공권 지원 제도는 무엇이며, 이와...
✅ [10/11] 동원예비군훈련의 종류와 훈련 시간은 어떻게 되며, 예비군 훈련 참가비는 ...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)\nYIELD node, score\nRETURN node.id AS id, node.name AS name, node.description AS description, score\nORDER BY score DESC"


✅ [11/11] 육·해·공·해병대 현역병이 대중교통 이용 중 상해사망 시 보장 한도는 얼...


In [38]:
# 4) 평가 실행 (ragas.evaluate 사용) + 결과 저장
result = run_evaluation(samples)
result_df = result.to_pandas()
result_df.to_csv("army_rag_eval_result.csv", index=False, encoding="utf-8-sig")

print("===== 평가 결과 =====")
print(result)
result_df


C:\Users\박동관\AppData\Local\Temp\ipykernel_54448\3211125010.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
C:\Users\박동관\AppData\Local\Temp\ipykernel_54448\3211125010.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(
Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3

===== 평가 결과 =====
{'faithfulness': 0.6807, 'answer_relevancy': 0.5371, 'llm_context_precision_with_reference': 0.4639, 'context_recall': 0.8788}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,병 복지의 혜택은 무엇인가요?,[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,"야, 병 복지 혜택은 진짜 많아. 일단 기본적으로는 급식이랑 의료 혜택이 잘 되어 ...",병 복지혜택 이렇게 다양합니다.,0.285714,0.000000,1.000000,1.000000
1,경상국립대에서 제공하는 건강검진의 이용대상은 누구인가요?,[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,"경상국립대에서 제공하는 건강검진의 이용대상은 현역 군인, 예비역, 그리고 군인공제회...",경상국립대에서 제공하는 건강검진의 이용대상은 현역/예비역/兵 회원 및 회원 가족(배...,0.500000,0.799211,0.500000,1.000000
2,차량야전정비병의 역할은 무엇인가요?,[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,차량야전정비병의 역할은 주로 군에서 사용하는 차량의 정비와 수리를 담당하는 거야. ...,"차량야전정비병은 군에서 차량의 유지보수 및 정비를 담당하는 병과로, 다양한 군용 차...",0.000000,0.384166,0.000000,0.000000
3,해군의 장기복무 선발 혜택은 무엇인가요?,[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,"해군의 장기복무 선발 혜택은 여러 가지가 있어. 우선, 장기복무에 선발되면 소령(상...","해군의 장기복무 선발 혜택으로는 소령(상사) 진출 기회, 계급별 정년(소령 50세,...",0.777778,0.634626,0.333333,1.000000
4,국외군사교육과 국내·외 학위과정 지원은 어떻게 전문인력 양성을 위한 기회를 제공하나요?,[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,국외군사교육과 국내·외 학위과정 지원은 전문인력 양성을 위한 중요한 기회를 제공해....,국외군사교육은 최신 군사지식을 습득하고 무관요원 및 권역별 지역전문가 양성을 위해 ...,1.000000,0.608067,0.638889,1.000000
5,취업컨설팅이 자기개발에 어떻게 도움을 줄 수 있나요?,[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,취업컨설팅은 자기개발에 많은 도움을 줄 수 있어. 이 서비스는 주로 1:1 전문 상...,취업컨설팅은 청년장병 및 단기간부를 대상으로 1:1 전문상담사를 배정하여 진로탐색과...,1.000000,0.788937,0.325000,0.666667
6,2026 병 복지 길라잡이에서 제공하는 군 e-러닝 학습 콘텐츠와 특기병 교육훈련 ...,[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,군 e-러닝 학습 콘텐츠와 특기병 교육훈련 이수결과의 학점 인정은 서로 연결되어 있...,"2026 병 복지 길라잡이에서는 군 e-러닝 학습 콘텐츠를 통해 외국어, 자격증, ...",0.750000,0.620350,0.645833,1.000000
7,"병역판정검사와 관련하여 나라사랑카드의 주요 기능은 무엇이며, 이 카드가 제공하는 서...",[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,"나라사랑카드는 병역판정검사와 관련하여 여러 주요 기능을 제공하며, 입대 전, 복무 ...","병역판정검사와 관련하여 나라사랑카드는 입대 전, 복무 중, 전역 후에 각각 다양한 ...",1.000000,0.635298,0.559524,1.000000
8,"인천에서 군인 및 군무원 가족을 위한 항공권 지원 제도는 무엇이며, 이와 관련된 할...",[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,"인천에서 군인 및 군무원 가족을 위한 항공권 지원 제도는 군인과 군무원, 그리고 그...","인천에서 군인 및 군무원 가족을 위한 항공권 지원 제도는 시민권자, 국외영주권자, ...",0.882353,0.747050,0.291667,1.000000
9,"동원예비군훈련의 종류와 훈련 시간은 어떻게 되며, 예비군 훈련 참가비는 얼마인가요?",[--- [참조 문서 1] ---\n▶▶▶ 병 복지혜택 이렇게 다양합니다\n2026...,동원예비군훈련은 크게 두 가지 종류로 나뉘어져 있어:\n\n1. **동원훈련 I형*...,"동원예비군훈련은 두 가지 유형으로 나뉘며, 동원훈련Ⅰ형은 예비군 1∼4년차 중 병력...",0.916667,0.690734,0.477778,1.000000
